# SDN on ME4OH full-field OFAM3 SST data

Experiment 4: Using SDN only to produce reconstructions

In [ ]:
# Standard imports
import json
import os

# Third-party imports
import numpy as np
import torch
from sklearn.preprocessing import MinMaxScaler

# Local imports
import models  # https://github.com/Jan-Williams/pyshred/blob/main/models.py 
from Data.preprocess_sst_ofam3_shred import (
    load_files,
    split_ordered_data,
)
from helper_functions.saving import JsonEncoder

np.random.seed(42)

## SHRED preprocessing

In [ ]:
# Experiment 4.1:
exp_index = 4
num_sensors = 5
test_lags = 52

In [ ]:
# Load and reformat data
data, dates, lats, longs = load_files()

In [ ]:
print("Preprocessing data...")
train_data, val_data, test_data, test_dates = split_ordered_data(data, dates)

# Normalise data
sc = MinMaxScaler()
sc = sc.fit(train_data)  # Computes min and max from training data only (prevent data leakage)
train_transformed = sc.transform(train_data)  
val_transformed = sc.transform(val_data)
test_transformed = sc.transform(test_data)

In [ ]:
class SDNDataset(torch.utils.data.Dataset):
    """TBC"""
    def __init__(self, X, Y):  # noqa: D107
        self.X = X
        self.Y = Y
        self.len = X.shape[0]
        
    def __getitem__(self, index):  # noqa: D105
        return self.X[index], self.Y[index]
    
    def __len__(self):  # noqa: D105
        return self.len

In [ ]:
def create_sdn_datasets(data, sensor_locs, test_lags=None):
    """TBC"""
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    if test_lags is None:
        # Not matching test dataset to SHRED test dataset
        data_in = np.array([data[i, sensor_locs] for i in range(len(data))])
        data_in = torch.tensor(data_in, dtype=torch.float32).to(device)
        data_out = torch.tensor(data, dtype=torch.float32).to(device)

    else:
        # Want only to test on the end point for each lag so can directly compare SDN and SHRED performance
        data_in = np.array([data[i+test_lags, sensor_locs] for i in range(len(data)-test_lags)])
        data_in = torch.tensor(data_in, dtype=torch.float32).to(device)
        ### -1 to have output be at the same time as final sensor measurements
        data_out = torch.tensor(data[np.array(range(len(data)-test_lags)) + test_lags - 1], dtype=torch.float32).to(device)

    dataset = SDNDataset(data_in, data_out)

    return dataset

In [ ]:
# Build dataset
sensor_locations = np.random.choice(data.shape[1], size=num_sensors, replace=False)
sensor_coords=(longs[sensor_locations], lats[sensor_locations])

train_dataset = create_sdn_datasets(train_transformed, sensor_locations)
val_dataset = create_sdn_datasets(val_transformed, sensor_locations)
test_dataset = create_sdn_datasets(test_transformed, sensor_locations)

In [ ]:
# Generate test set to match SHRED test set
# (use final sensor results for what each sequence would be if using SHRED)
test_dataset_match_shred = create_sdn_datasets(test_transformed, sensor_locations, test_lags=test_lags)
test_lag_dates = test_dates[np.array(range(len(test_dates)-test_lags)) + test_lags - 1]

## Train + test SDN model

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
sdn_model = models.SDN(num_sensors, len(lats)).to(device)
validation_errors = models.fit(sdn_model, train_dataset, val_dataset, batch_size=64,
                               num_epochs=1000, lr=1e-3, verbose=True, patience=5)

In [ ]:
test_recons = sc.inverse_transform(sdn_model(test_dataset.X).detach().cpu().numpy())
test_ground_truth = sc.inverse_transform(test_dataset.Y.detach().cpu().numpy())
print('Test Reconstruction Error: ')
print(np.linalg.norm(test_recons - test_ground_truth) / np.linalg.norm(test_ground_truth))

In [ ]:
test_recons_match_shred = sc.inverse_transform(sdn_model(test_dataset_match_shred.X).detach().cpu().numpy())
ground_truth_match_shred = sc.inverse_transform(test_dataset_match_shred.Y.detach().cpu().numpy())
print('Test Reconstruction Error: ')
print(np.linalg.norm(test_recons_match_shred - ground_truth_match_shred)/ np.linalg.norm(ground_truth_match_shred))

## Save results

In [ ]:
# Make experiment folder
directory = f"Reconstructions/Static/Exp{exp_index}/"
os.makedirs(directory, exist_ok = True)

In [ ]:
np.save(f"Reconstructions/Static/Exp{exp_index}/recons_n{num_sensors}", test_recons)
np.save(f"Reconstructions/Static/Exp{exp_index}/truth_n{num_sensors}", test_ground_truth)


np.save(f"Reconstructions/Static/Exp{exp_index}/recons_n{num_sensors}_l{test_lags}", test_recons_match_shred)
np.save(f"Reconstructions/Static/Exp{exp_index}/truth_n{num_sensors}_l{test_lags}", ground_truth_match_shred)

In [ ]:
metadata = {
    "sensor_locations": list(sensor_locations),
    "sensor_coords": list(sensor_coords),
    "test_dates": list(test_dates),
    "test_lag_dates": list(test_lag_dates)
    }

In [ ]:
with open(f"Reconstructions/Static/Exp{exp_index}/metadata_n{num_sensors}.json",
          'w', encoding='utf-8') as f:
    json.dump(metadata, f, cls=JsonEncoder)